# 01. Baseline: Rule Based Archetype Labels

Builds the page level feature table and assigns each page a rule based archetype using explicit thresholds. This is the independent ground truth every other notebook in this project checks its claims against. It does not use KMeans or any clustering. This is also the "claim before" a model is ever run, revisited in the model validation notebook.

## Setup

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')") 
# force single threaded aggregation. parallel SUM and AVG over the remote parquet row groups
# can sum in a different order each run, changing feature values enough to shift downstream
# results. single threaded execution makes page_level fully reproducible
con.execute("SET threads=1")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
import os
import pandas as pd
import numpy as np

DEFAULT_MIN_IMPRESSIONS = 10
DEFAULT_STALE_THRESHOLD_DAYS = 180
DEFAULT_LOW_ENGAGEMENT_THRESHOLD = 0.3
DEFAULT_GOOD_POSITION_MAX = 10

EXPECTED_CTR_BY_BUCKET = {
    "1_pos_1-3": 0.027805,
    "2_pos_4-10": 0.004229,
    "3_pos_11-20": 0.004184,
    "4_pos_21plus": 0.002788,
}

# shared by this notebook's rule based action and the clustering notebook's cluster
# naming, so the same archetype always maps to the same action everywhere in the project
ACTION_MAP = {
    "champions": "protect",
    "hidden_gems": "improve",
    "rising_stars": "monitor",
    "stale_visible_pages": "refresh",
    "engagement_problem_pages": "rewrite",
    "weak_no_demand_pages": "prune",
    "cannibalization_risk": "merge",
    "monitor": "monitor",
}


def position_bucket(pos):
    """Bucket an average ranking position into one of four expected-CTR tiers."""
    if pos is None or pd.isna(pos):
        return None
    if pos <= 3:
        return "1_pos_1-3"
    elif pos <= 10:
        return "2_pos_4-10"
    elif pos <= 20:
        return "3_pos_11-20"
    else:
        return "4_pos_21plus"


def assign_archetype(
    row,
    high_volume_impressions,
    min_impressions=DEFAULT_MIN_IMPRESSIONS,
    stale_threshold_days=DEFAULT_STALE_THRESHOLD_DAYS,
    low_engagement_threshold=DEFAULT_LOW_ENGAGEMENT_THRESHOLD,
    good_position_max=DEFAULT_GOOD_POSITION_MAX,
):
    """Assign one rule based archetype to a single page level row.

    high_volume_impressions is the 75th percentile of total_impressions across the full
    page level population, computed once by the caller and passed in here, since it is a
    property of the dataset, not a fixed threshold.
    """
    gsc_available = bool(row["gsc_data_available_any"]) if pd.notna(row["gsc_data_available_any"]) else False
    if (not gsc_available) or row["total_impressions"] < min_impressions:
        return "INSUFFICIENT_DATA"

    # negative days_since_last_update is a reporting lag artifact, not real freshness
    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] < 0:
        return "INSUFFICIENT_DATA"

    is_good_position = pd.notna(row["avg_position"]) and row["avg_position"] <= good_position_max
    is_high_volume = row["total_impressions"] >= high_volume_impressions
    is_low_ctr = pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0
    is_stale = pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > stale_threshold_days
    has_low_engagement = pd.notna(row["engagement_rate"]) and row["engagement_rate"] < low_engagement_threshold
    is_no_demand = row["total_impressions"] < min_impressions * 3
    ga4_available = bool(row["ga4_data_available_any"]) if pd.notna(row["ga4_data_available_any"]) else False
    is_cannibalization = bool(row["cannibalization_risk"]) if pd.notna(row["cannibalization_risk"]) else False

    if is_cannibalization:
        return "cannibalization_risk"
    if is_good_position and is_high_volume and not is_low_ctr and not is_stale:
        return "champions"
    if is_good_position and is_high_volume and is_stale:
        return "stale_visible_pages"
    if ga4_available and is_good_position and is_high_volume and has_low_engagement:
        return "engagement_problem_pages"
    if is_good_position and not is_high_volume and not is_low_ctr:
        return "hidden_gems"
    if not is_good_position and pd.notna(row["ctr_gap"]) and row["ctr_gap"] < 0 and row["total_impressions"] >= high_volume_impressions * 0.25:
        return "rising_stars"
    if is_no_demand:
        return "weak_no_demand_pages"

    return "monitor"


def action_score(
    row,
    high_volume_impressions,
    stale_threshold_days=DEFAULT_STALE_THRESHOLD_DAYS,
    low_engagement_threshold=DEFAULT_LOW_ENGAGEMENT_THRESHOLD,
):
    """Transparent, additive priority score. Every term is a fixed point value, nothing
    here is fit or learned, so the score stays auditable by reading this function."""
    score = 0.0
    if pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0:
        score += row["ctr_gap"] * 100  # bigger position adjusted gap, more clicks being left on the table
    if row["total_impressions"] >= high_volume_impressions:
        score += 2  # more traffic riding on the page raises the stakes of the call
    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > stale_threshold_days:
        score += 1  # stale content is a concrete, fixable problem
    if pd.notna(row["engagement_rate"]) and row["engagement_rate"] < low_engagement_threshold:
        score += 1  # traffic that does not engage despite ranking is a concrete, fixable problem
    if row["cannibalization_risk"]:
        score += 2  # two pages splitting one keyword is the most directly fixable waste here
    return round(score, 2)


def reason_codes(
    row,
    high_volume_impressions,
    stale_threshold_days=DEFAULT_STALE_THRESHOLD_DAYS,
    low_engagement_threshold=DEFAULT_LOW_ENGAGEMENT_THRESHOLD,
    good_position_max=DEFAULT_GOOD_POSITION_MAX,
):
    """Plain-word codes for which action_score conditions fired for a page."""
    codes = []
    if pd.notna(row["avg_position"]) and row["avg_position"] <= good_position_max:
        codes.append("good_position")
    if row["total_impressions"] >= high_volume_impressions:
        codes.append("high_volume")
    if pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0:
        codes.append("ctr_below_expected")
    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > stale_threshold_days:
        codes.append("stale")
    if pd.notna(row["engagement_rate"]) and row["engagement_rate"] < low_engagement_threshold:
        codes.append("low_engagement")
    if row["cannibalization_risk"]:
        codes.append("cannibalization")
    return ", ".join(codes) if codes else "no_flags"

## Step 1: Build the page level feature table

Pulling one row per client and content page from the warehouse means summing GSC impressions, clicks, and average position, and GA4 sessions and engaged sessions, over the full reporting window, then joining in the content metadata needed for staleness later. Every downstream step, the rule based baseline, the clustering, and the validation, needs one clean row per page, so building it once here keeps that logic in a single place instead of repeating it. 394928 rows come out at the page level, one per client and content pair.

In [3]:
page_level = con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    fact_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(ga4_sessions) AS total_sessions,
            SUM(ga4_engaged_sessions) AS total_engaged_sessions,
            BOOL_OR(gsc_data_available) AS gsc_data_available_any,
            BOOL_OR(ga4_data_available) AS ga4_data_available_any
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    content_meta AS (
        SELECT
            content_hash_id,
            keyword_hash_id,
            content_updated_date,
            is_published,
            is_deleted
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published = true AND is_deleted = false
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        cm.keyword_hash_id,
        f.total_impressions,
        f.total_clicks,
        f.avg_position,
        f.total_sessions,
        f.total_engaged_sessions,
        f.gsc_data_available_any,
        f.ga4_data_available_any,
        DATE_DIFF('day', cm.content_updated_date, ref.max_date) AS days_since_last_update,
        CASE WHEN f.total_impressions > 0 THEN f.total_clicks * 1.0 / f.total_impressions ELSE NULL END AS ctr,
        CASE WHEN f.total_sessions > 0 THEN f.total_engaged_sessions * 1.0 / f.total_sessions ELSE NULL END AS engagement_rate
    FROM fact_agg f
    JOIN content_meta cm ON f.content_hash_id = cm.content_hash_id
    CROSS JOIN ref
""").df()

print(page_level.shape)
page_level.head()

(394928, 13)


,client_hash_id,content_hash_id,keyword_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_engaged_sessions,gsc_data_available_any,ga4_data_available_any,days_since_last_update,ctr,engagement_rate
0,client_3ffa76342f366962,content_1a6296faee432dae,keyword_5f9f0577c4c18307,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
1,client_3ffa76342f366962,content_73f21e612565035a,keyword_41684cca996c95a7,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
2,client_3ffa76342f366962,content_5a5be514ff559598,keyword_66ba2c2dfa653eed,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
3,client_3ffa76342f366962,content_05b377d0c8a5cfd8,keyword_935dbe1165583ac2,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN
4,client_3ffa76342f366962,content_dc34c661d63e55a9,keyword_d9e7f10c088179f0,0.0,0.0,NaN,0.0,0.0,False,False,41,NaN,NaN


## Step 2: CTR gap against the expected CTR by position bucket, and the cannibalization flag

Each page gets bucketed by its average position, an expected CTR for that bucket is looked up from an earlier verified query, and the gap between expected and actual CTR gets stored. CTR only means something relative to where a page ranks, a 1 percent CTR is weak at position 3 and strong at position 25, so this position adjusted gap is what several archetype rules and the clustering naming confidence checks compare against later, not raw CTR. The same pass counts how many pages at a given client rank for the same keyword and flags anything with more than one as a cannibalization risk, a fact clustering cannot see on its own since it depends on two separate pages sharing a client and a keyword, not on any single page's own metrics. Every page now carries a ctr_gap and a cannibalization_risk flag.

In [4]:
page_level["position_bucket"] = page_level["avg_position"].apply(position_bucket)
page_level["expected_ctr"] = page_level["position_bucket"].map(EXPECTED_CTR_BY_BUCKET)
page_level["ctr_gap"] = page_level["expected_ctr"] - page_level["ctr"]

In [5]:
keyword_counts = (
    page_level[page_level["total_impressions"] > 0]
    .groupby(["client_hash_id", "keyword_hash_id"])["content_hash_id"]
    .nunique()
    .reset_index(name="pages_ranking_for_keyword")
)
page_level = page_level.merge(keyword_counts, on=["client_hash_id", "keyword_hash_id"], how="left")
page_level["cannibalization_risk"] = page_level["pages_ranking_for_keyword"].fillna(0) > 1

## Step 3: Rule based archetype labels, the claim before any model is run

Each page gets assigned one of the assignment's named archetypes, or monitor, using explicit thresholds on position, volume, CTR gap, staleness, and engagement. This is the independent, hand written claim about what archetypes exist, built before KMeans ever sees the data, and the model validation notebook checks whether the model's own claim, made after running KMeans, agrees with this one. Of the pages with usable data, monitor is by far the largest group, followed by rising_stars, weak_no_demand_pages, champions, engagement_problem_pages, hidden_gems, and a small handful of stale_visible_pages and cannibalization_risk pages.

In [6]:
MIN_IMPRESSIONS = 10
STALE_THRESHOLD_DAYS = 180
HIGH_VOLUME_IMPRESSIONS = page_level["total_impressions"].quantile(0.75)

page_level["archetype"] = page_level.apply(
    lambda row: assign_archetype(row, HIGH_VOLUME_IMPRESSIONS, MIN_IMPRESSIONS, STALE_THRESHOLD_DAYS),
    axis=1,
)
print("claim before, the rule based archetype distribution:")
page_level["archetype"].value_counts()

claim before, the rule based archetype distribution:


archetype
INSUFFICIENT_DATA           272549
monitor                      56390
rising_stars                 19211
weak_no_demand_pages         17426
champions                    14300
engagement_problem_pages     12752
hidden_gems                   2271
stale_visible_pages             27
cannibalization_risk             2
Name: count, dtype: int64

## Step 4: A transparent score, reason codes, and a ranked list

archetype alone says which bucket a page landed in, not how strongly it belongs there or which pages inside a bucket deserve attention first. action_score fixes that with a plain additive point total built from the same signals Step 3 already computes: the position adjusted CTR gap when a page is underperforming for its rank, extra weight for high volume pages since more traffic raises the stakes of getting the call right, one point for staleness, one point for low engagement despite traffic, and two points for a confirmed cannibalization conflict since two pages splitting one keyword is the most directly fixable waste in this data. Nothing here is fit or learned, every term is a fixed number chosen for how directly actionable that signal is, so the score stays auditable by reading the function below rather than by trusting it. reason_codes writes out in plain words which of those conditions actually fired for a page, and action turns each archetype into one of six calls to action reused later for the model's own clusters: protect, improve, rewrite, merge, prune, or monitor. Sorting the usable pages, meaning everything except INSUFFICIENT_DATA, by action_score descending gives the ranked list the next two steps check.

In [7]:
page_level["action_score"] = page_level.apply(lambda row: action_score(row, HIGH_VOLUME_IMPRESSIONS, STALE_THRESHOLD_DAYS), axis=1)
page_level["reason_codes"] = page_level.apply(lambda row: reason_codes(row, HIGH_VOLUME_IMPRESSIONS, STALE_THRESHOLD_DAYS), axis=1)
page_level["action"] = page_level["archetype"].map(ACTION_MAP)

usable = page_level[page_level["archetype"] != "INSUFFICIENT_DATA"].copy()
usable["actionable"] = usable["archetype"] != "monitor"
ranked = usable.sort_values("action_score", ascending=False).reset_index(drop=True)

print(f"{len(usable)} usable pages ranked by action_score")
ranked[["client_hash_id", "content_hash_id", "archetype", "action", "reason_codes", "action_score"]].head(10)

122379 usable pages ranked by action_score


,client_hash_id,content_hash_id,archetype,action,reason_codes,action_score
0,client_73cda7b4e4f265ea,content_391ba4add35aaa11,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
1,client_20259bd6705d81d4,content_d4adef9d6fac9b8c,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
2,client_20259bd6705d81d4,content_e10ff20474447d10,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
3,client_20259bd6705d81d4,content_860b03f8141381ef,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
4,client_20259bd6705d81d4,content_d1362e0d4637b4ca,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
5,client_86ebc2f12c01f586,content_3d8e9adec1caa9e5,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
6,client_23a62021009f63c4,content_133f998472a88bab,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
7,client_e00b29e582949543,content_abd650f81fd64146,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
8,client_e00b29e582949543,content_ea6e9ede8d6dd164,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78
9,client_a80fca3f171ed1de,content_be693cdcc2d886e9,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78


## Step 5: Precision@K against the base rate and two weaker baselines

There is no outside ground truth for whether a page really deserved attention, this notebook is the ground truth every other notebook in this project checks its claims against, so precision@K here is not validation against reality, it is a self consistency check: does sorting by action_score front load pages the rule engine already calls actionable, meaning any archetype other than monitor, more than doing nothing or doing something naive would. actionable_base_rate is the share of usable pages that are actionable with no ranking at all, the floor any ranking has to clear to be worth the trouble of computing. random_shuffle reorders the same pages with no signal at all, its precision@K should land close to the base rate by construction, and checking that empirically is a test of the check itself rather than an assumption taken on faith. naive_traffic_ranked sorts purely by total_impressions, the ranking an analyst reaches for without building any rules, a floor below the floor since it uses real signal but none of the position, staleness, or engagement context action_score uses. action_score_ranked is what Step 4 produced.

In [8]:
K_VALUES = [20, 50, 100, 500, 1000]

actionable_base_rate = usable["actionable"].mean()

random_shuffle = usable.sample(frac=1, random_state=42).reset_index(drop=True)
naive_traffic = usable.sort_values("total_impressions", ascending=False).reset_index(drop=True)

precision_at_k = pd.DataFrame({
    "K": K_VALUES,
    "action_score_ranked": [ranked.head(k)["actionable"].mean() for k in K_VALUES],
    "naive_traffic_ranked": [naive_traffic.head(k)["actionable"].mean() for k in K_VALUES],
    "random_shuffle": [random_shuffle.head(k)["actionable"].mean() for k in K_VALUES],
})
precision_at_k["actionable_base_rate"] = round(actionable_base_rate, 3)
precision_at_k

,K,action_score_ranked,naive_traffic_ranked,random_shuffle,actionable_base_rate
0,20,1.000,0.700,0.650,0.539
1,50,1.000,0.760,0.600,0.539
2,100,1.000,0.830,0.490,0.539
3,500,1.000,0.844,0.528,0.539
4,1000,0.624,0.850,0.539,0.539


## Step 6: Top 20, hand reviewed

Precision@K says the ranking beats the floor, reading the actual pages is what says whether the top of that ranking makes sense. The next cell prints the 20 highest action_score pages with their archetype, action, and reason codes, the hand review underneath was written from those printed rows, not from a general description of what should be there.

In [9]:
review_cols = [
    "client_hash_id", "content_hash_id", "archetype", "action", "reason_codes", "action_score",
    "total_impressions", "avg_position", "ctr_gap", "days_since_last_update", "engagement_rate",
]
top_20 = ranked[review_cols].head(20)
top_20

,client_hash_id,content_hash_id,archetype,action,reason_codes,action_score,total_impressions,avg_position,ctr_gap,days_since_last_update,engagement_rate
0,client_73cda7b4e4f265ea,content_391ba4add35aaa11,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,482.0,2.201560,0.027805,125,0.0
1,client_20259bd6705d81d4,content_d4adef9d6fac9b8c,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,272.0,2.699412,0.027805,41,0.0
2,client_20259bd6705d81d4,content_e10ff20474447d10,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,272.0,2.314083,0.027805,41,0.0
3,client_20259bd6705d81d4,content_860b03f8141381ef,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,341.0,2.721662,0.027805,41,0.0
4,client_20259bd6705d81d4,content_d1362e0d4637b4ca,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,305.0,2.783104,0.027805,41,0.0
5,client_86ebc2f12c01f586,content_3d8e9adec1caa9e5,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,635.0,2.694199,0.027805,41,0.0
6,client_23a62021009f63c4,content_133f998472a88bab,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,209.0,2.725656,0.027805,41,0.0
7,client_e00b29e582949543,content_abd650f81fd64146,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,204.0,2.837075,0.027805,41,0.0
8,client_e00b29e582949543,content_ea6e9ede8d6dd164,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,225.0,2.371541,0.027805,41,0.0
9,client_a80fca3f171ed1de,content_be693cdcc2d886e9,engagement_problem_pages,rewrite,"good_position, high_volume, ctr_below_expected...",5.78,546.0,2.586838,0.027805,29,0.0


All 20 rows tie at an action_score of 5.78, and reading the columns explains why rather than leaving it as a coincidence: every one of these pages ranks in position 1 to 3, the top expected CTR bucket, carries a ctr_gap of exactly 0.027805, meaning its actual CTR is exactly zero, and its engagement_rate is exactly zero too. Ranking at the top of page one with real impressions, 123 to 673 of them, and getting not one click and not one engaged session is the single most extreme combination the scoring function can hit, high_volume plus the full position 1 to 3 CTR gap plus low_engagement, so a wide tie at the top is the correct behavior of a coarse, transparent score meeting a genuinely unusual pocket of pages, not a bug to fix.

Four clients account for 14 of these 20 rows, client_20259bd6705d81d4 five times, and client_e00b29e582949543, client_1a8bf67cad4ee525, and client_0fa64a184f18a4a0 three times each. That concentration is worth naming plainly: this top-20 slice is closer to four client level stories than twenty independent page level ones, the same client domination check the accompanying paper's limitations section asks a reviewer to run before trusting a cluster gets applied here too. A zero CTR at position 1 to 3 with decent volume is also unusual enough on its own to question before acting on it. It is consistent with a broken or missing title and meta description, a SERP feature such as a featured snippet or knowledge panel absorbing the click, or a branded and navigational query where a real person does not need to click through, and it is just as consistent with a GSC tracking gap that has nothing to do with the page at all. rewrite is the right action to queue, confirming which of those explanations fits is not something this table can do by itself, it is exactly the kind of manual check the accompanying paper's limitations section asks a person to make before acting.

precision_at_k also is not a flat win at every depth. It holds at a perfect 1.0 through K=500, then drops to 0.624 at K=1000, still well above the 0.539 base rate and the naive traffic ranking at that depth, but a real drop, not a rounding artifact. The score's near perfect run at the top reflects how sharply defined that zero CTR, zero engagement, top position pocket is, not a guarantee that ranking stays this clean a thousand rows deep.

## Save output

page_level, with its archetype, cannibalization_risk, action_score, reason_codes, and action columns, gets written here to a shared interim file. The clustering notebook loads this directly instead of re querying the warehouse, so every downstream notebook works from the exact same baseline claim. work/interim/page_level.csv gets written for the clustering notebook to load.

In [10]:
os.makedirs("../interim", exist_ok=True)
page_level.to_csv("../interim/page_level.csv", index=False)
print(f"wrote {len(page_level)} rows to work/interim/page_level.csv")

wrote 394928 rows to work/interim/page_level.csv
